# Testbenches and waveforms (Verilog and CocoTB)

**Goal:** write a testbench, run it, and read the waveform to see what the hardware did.

```
design + testbench  ->  simulator (Icarus Verilog)  ->  signal changes  ->  waveform picture
```

Part A uses a Verilog testbench. Part B uses a Python testbench (CocoTB). Run the cells from top to bottom with **Shift + Enter**.

## Step 0: Install the tools (run once)
On Colab this installs Icarus Verilog and CocoTB automatically. On your own computer it only checks whether they are there.
You should see three lines starting with `[OK]`.

In [ ]:
import os, sys, shutil, subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)

# 1. Icarus Verilog (auto-installs on Colab/Linux)
if shutil.which("iverilog") is None:
    if sys.platform.startswith("linux"):
        print("Installing Icarus Verilog (about 30 seconds)...")
        rc, out = sh("apt-get update -qq && apt-get install -y -qq iverilog")
        if rc != 0:
            print("Icarus install failed. Last lines of the error:")
            print(out[-800:])
    else:
        print("Install Icarus Verilog for your computer, then restart VSCode.")
        print("Windows: https://bleyer.org/icarus/   Mac: brew install icarus-verilog")

# 2. CocoTB and matplotlib (installs into the Python this notebook uses)
rc, out = sh(f'"{sys.executable}" -m pip install -q cocotb matplotlib')
if rc != 0:
    print("pip install failed. Last lines of the error:")
    print(out[-800:])

# 3. Check everything
print("---- setup check ----")
ic = shutil.which("iverilog")
print(("[OK]  " if ic else "[X]   ") + "Icarus Verilog", "-", sh("iverilog -V")[1].splitlines()[0] if ic else "not found")

rc, out = sh(f'"{sys.executable}" -c "import cocotb; print(cocotb.__version__)"')
print(("[OK]  " if rc == 0 else "[X]   ") + "CocoTB", "-", out.strip() if rc == 0 else "not installed")

rc, out = sh(f'"{sys.executable}" -c "import matplotlib; print(matplotlib.__version__)"')
print(("[OK]  " if rc == 0 else "[X]   ") + "matplotlib", "-", out.strip() if rc == 0 else "not installed")

if os.environ.get("IVERILOG_DUMPER"):
    print("Note: IVERILOG_DUMPER is set to", os.environ["IVERILOG_DUMPER"], "- this can switch waveform dumping off.")

## Step 1: Helpers (just run this cell)
- `run_sim(...)` runs a Verilog testbench.
- `run_cocotb(...)` runs a Python (CocoTB) testbench.
- `show_waves(...)` draws the waveform.

You do not need to read the code inside.

In [ ]:
import os, sys, shutil, subprocess

# ---------- Verilog testbench flow ----------
def run_sim(design, testbench, top="tb", name="sim"):
    """Compile and run a design with a Verilog testbench. Returns the waveform file path."""
    if shutil.which("iverilog") is None:
        print("iverilog is missing. Run Step 0 again.")
        return None
    os.makedirs("build", exist_ok=True)
    d, t, vvp = f"build/{name}_design.sv", f"build/{name}_tb.sv", f"build/{name}.vvp"
    with open(d, "w") as f:
        f.write(design)
    with open(t, "w") as f:
        f.write(testbench)
    if os.path.exists("build/dump.vcd"):
        os.remove("build/dump.vcd")
    r = subprocess.run(["iverilog", "-g2012", "-s", top, "-o", vvp, d, t],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("COMPILE ERROR (fix this, then run the cell again):")
        print(r.stderr or r.stdout)
        return None
    if r.stderr.strip():
        print(r.stderr)
    env = dict(os.environ, IVERILOG_DUMPER="vcd")
    r = subprocess.run(["vvp", vvp, "-vcd"], capture_output=True, text=True, env=env)
    print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)
    if not os.path.exists("build/dump.vcd"):
        if "suppressed" in r.stdout:
            print("Icarus has waveform dumping switched off in this session.")
        print("No waveform file was written. Does your testbench call $dumpfile and $dumpvars?")
        return None
    return "build/dump.vcd"


# ---------- CocoTB (Python testbench) flow ----------
# This small helper is saved next to your test. Calling wave_recorder.start(dut)
# at the top of a test records every signal change, so the waveform does not
# depend on Icarus writing a .vcd file.
_RECORDER = """
import os
import cocotb
from cocotb.triggers import Edge
from cocotb.utils import get_sim_time

_log = None

def _write(name, value):
    _log.write("%d %s %s" % (int(get_sim_time("ps")), name, str(value).lower()))
    _log.write(chr(10))
    _log.flush()

async def _watch(name, handle):
    while True:
        await Edge(handle)
        _write(name, handle.value)

def start(dut):
    global _log
    path = os.environ.get("WAVE_LOG")
    if not path:
        return
    _log = open(path, "w")
    for handle in dut:
        try:
            value = handle.value
        except Exception:
            continue
        name = getattr(handle, "_name", None) or str(handle)
        _write(name, value)
        cocotb.start_soon(_watch(name, handle))
"""

_RUNNER = """
import sys
sys.path.insert(0, __WORKDIR__)
try:
    from cocotb_tools.runner import get_runner   # CocoTB 2.x
except ImportError:
    from cocotb.runner import get_runner         # CocoTB 1.x

runner = get_runner("icarus")
srcs = [__DESIGN__]
try:
    runner.build(sources=srcs, hdl_toplevel=__TOP__,
                 build_args=["-g2012"], build_dir=__BUILD__, always=True)
except TypeError:
    runner.build(verilog_sources=srcs, hdl_toplevel=__TOP__,
                 build_args=["-g2012"], build_dir=__BUILD__, always=True)
runner.test(hdl_toplevel=__TOP__, test_module="test_dut", build_dir=__BUILD__)
"""

def run_cocotb(design, test_code, top, name="cocotb"):
    """Run a CocoTB test (Python) against a Verilog design. Returns the waveform path."""
    if shutil.which("iverilog") is None:
        print("iverilog is missing. Run Step 0 again.")
        return None
    work = os.path.abspath(os.path.join("build", name))
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work)
    log = os.path.join(work, "waves.log")
    design_path = os.path.join(work, "design.sv")
    with open(design_path, "w") as f:
        f.write("`timescale 1ns/1ps" + chr(10) + design)
    with open(os.path.join(work, "wave_recorder.py"), "w") as f:
        f.write(_RECORDER)
    with open(os.path.join(work, "test_dut.py"), "w") as f:
        f.write(test_code)
    script = (_RUNNER.replace("__WORKDIR__", repr(work))
                     .replace("__DESIGN__", repr(design_path))
                     .replace("__TOP__", repr(top))
                     .replace("__BUILD__", repr(os.path.join(work, "sim_build"))))
    script_path = os.path.join(work, "run_all.py")
    with open(script_path, "w") as f:
        f.write(script)
    env = dict(os.environ, COCOTB_ANSI_OUTPUT="0", PYTHONUNBUFFERED="1", WAVE_LOG=log)
    r = subprocess.run([sys.executable, script_path], capture_output=True, text=True,
                       cwd=work, env=env)
    out = (r.stdout or "") + (r.stderr or "")
    print(out[-3500:])
    xml = os.path.join(work, "sim_build", "results.xml")
    if os.path.exists(xml):
        text = open(xml).read()
        n_tests, n_fail = text.count("<testcase"), text.count("<failure")
        print("RESULT:", "PASS" if n_fail == 0 and n_tests > 0 else "FAIL",
              f"({n_tests - n_fail} of {n_tests} tests passed)")
    else:
        print("RESULT: no results file. The build or the test did not finish; read the messages above.")
    if os.path.exists(log) and os.path.getsize(log) > 0:
        return log
    print("No waveform was recorded. Does your test call wave_recorder.start(dut) first?")
    return None


# ---------- Waveform drawing ----------
def read_log(path):
    """Read the waveform log written by wave_recorder. Returns (signals, timescale)."""
    sigs = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 3:
                continue
            t, name, val = int(parts[0]), parts[1], parts[2]
            s = sigs.setdefault(name, {"name": name, "width": len(val), "ch": []})
            if not s["ch"] or s["ch"][-1][1] != val:
                s["ch"].append((t, val))
    return sigs, "ps"


def read_vcd(path):
    """Read a .vcd file. Returns (signals, timescale)."""
    with open(path) as f:
        toks = f.read().split()
    sigs, scope, timescale = {}, [], ""
    t, i, n, defs_done = 0, 0, len(toks), False
    while i < n:
        tk = toks[i]
        if tk in ("$date", "$version", "$comment", "$timescale"):
            j = i + 1
            while toks[j] != "$end":
                j += 1
            if tk == "$timescale":
                timescale = "".join(toks[i + 1:j])
            i = j + 1
        elif tk == "$scope":
            scope.append(toks[i + 2])
            i += 4
        elif tk == "$upscope":
            scope.pop()
            i += 2
        elif tk == "$var":
            width, ident, name = int(toks[i + 2]), toks[i + 3], toks[i + 4]
            j = i + 5
            while toks[j] != "$end":
                j += 1
            label = ".".join(scope[1:] + [name]) if len(scope) > 1 else name
            if ident not in sigs:
                sigs[ident] = {"name": label, "width": width, "ch": []}
            i = j + 1
        elif tk == "$enddefinitions":
            defs_done = True
            i += 2
        elif not defs_done or tk.startswith("$"):
            i += 1
        elif tk[0] == "#":
            t = int(tk[1:])
            i += 1
        elif tk[0] in "bB":
            if toks[i + 1] in sigs:
                sigs[toks[i + 1]]["ch"].append((t, tk[1:].lower()))
            i += 2
        elif tk[0] in "rR":
            i += 2
        else:
            if tk[1:] in sigs:
                sigs[tk[1:]]["ch"].append((t, tk[0].lower()))
            i += 1
    return sigs, timescale


def _fmt(v, radix):
    if not set(v) <= {"0", "1"}:
        return "X" if "x" in v else "Z"
    if radix == "bin":
        return v
    if radix == "dec":
        return str(int(v, 2))
    return format(int(v, 2), "X")


def _value_at(ch, t):
    val = None
    for tt, v in ch:
        if tt <= t:
            val = v
        else:
            break
    return val


def show_waves(wave_path="build/dump.vcd", signals=None, t_max=None,
               radix="hex", cursor=None, theme="dark"):
    """Draw the waveform like Vivado's viewer.
    signals = names to show (optional)   radix = "hex", "dec" or "bin"
    cursor  = time to read values at, in ns (default: the end)   theme = "dark" or "light"
    """
    import re
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec
    if str(wave_path).endswith(".log"):
        sigs, ts = read_log(wave_path)
    else:
        sigs, ts = read_vcd(wave_path)
    items = [s for s in sigs.values() if s["ch"]]
    if signals:
        order = {name: k for k, name in enumerate(signals)}
        items = sorted([s for s in items if s["name"] in order], key=lambda s: order[s["name"]])
    if not items:
        print("No signals to draw. Names available:", [s["name"] for s in sigs.values()])
        return

    m = re.match(r"(\d*)\s*(fs|ps|ns|us|ms|s)", ts or "")
    per = {"fs": 1e-6, "ps": 1e-3, "ns": 1.0, "us": 1e3, "ms": 1e6, "s": 1e9}
    scale = int(m.group(1) or 1) * per[m.group(2)] if m else 1.0
    unit = "ns" if m else "time units"

    if theme == "light":
        bg, panel, fg, grid, wave, red, cur_c = "white", "#f1f1f1", "#202020", "#dddddd", "#0a7d0a", "#d62020", "#c89600"
    else:
        bg, panel, fg, grid, wave, red, cur_c = "#0e0e0e", "#1c1c1c", "#e6e6e6", "#2c2c2c", "#2ee62e", "#ff4545", "#ffd400"

    end_raw = t_max or max(t for s in items for t, _ in s["ch"])
    end = max(end_raw, 1) * scale
    cur = (cursor if cursor is not None else end_raw * scale)
    cur_raw = cur / scale
    n = len(items)

    fig = plt.figure(figsize=(11, 0.6 * n + 1.1), facecolor=bg)
    gs = GridSpec(n, 3, width_ratios=[1.5, 1.0, 9], hspace=0, wspace=0,
                  left=0.01, right=0.995, top=0.86, bottom=0.2)
    first_wave = None
    for r, s in enumerate(items):
        axn = fig.add_subplot(gs[r, 0])
        axv = fig.add_subplot(gs[r, 1])
        axw = fig.add_subplot(gs[r, 2], sharex=first_wave) if first_wave else fig.add_subplot(gs[r, 2])
        first_wave = first_wave or axw
        for a in (axn, axv):
            a.set_facecolor(panel)
            a.set_xticks([]); a.set_yticks([])
            for sp in a.spines.values():
                sp.set_color(grid)
        axn.text(0.06, 0.5, s["name"], color=fg, va="center", ha="left",
                 transform=axn.transAxes, fontsize=10)
        val = _value_at(s["ch"], cur_raw)
        axv.text(0.5, 0.5, _fmt(val, radix) if val is not None else "", color=wave if val and set(val) <= {"0", "1"} else red,
                 va="center", ha="center", transform=axv.transAxes, fontsize=10)

        axw.set_facecolor(bg)
        axw.set_ylim(-0.2, 1.2)
        axw.set_yticks([])
        axw.grid(axis="x", color=grid, linewidth=0.6)
        for side in ("top", "right", "left"):
            axw.spines[side].set_visible(False)
        axw.spines["bottom"].set_color(grid)

        ch = [(t * scale, v) for t, v in s["ch"] if t * scale <= end]
        for k, (t0, v) in enumerate(ch):
            t1 = ch[k + 1][0] if k + 1 < len(ch) else end
            known = set(v) <= {"0", "1"}
            col = wave if known else red
            if s["width"] == 1:
                if known:
                    y = int(v)
                    axw.plot([t0, t1], [y, y], color=col, linewidth=1.6)
                    if k + 1 < len(ch) and set(ch[k + 1][1]) <= {"0", "1"}:
                        y2 = int(ch[k + 1][1])
                        if y2 != y:
                            axw.plot([t1, t1], [y, y2], color=col, linewidth=1.6)
                else:
                    axw.fill_between([t0, t1], 0, 1, color=red, alpha=0.55, linewidth=0)
            else:
                d = min(end * 0.004, (t1 - t0) / 2)
                axw.plot([t0, t0 + d, t1 - d, t1, t1 - d, t0 + d, t0],
                         [0.5, 1, 1, 0.5, 0, 0, 0.5], color=col, linewidth=1.3)
                text = _fmt(v, radix)
                if (t1 - t0) / end > 0.011 * len(text) + 0.008:
                    axw.text((t0 + t1) / 2, 0.5, text, color=fg, ha="center", va="center", fontsize=9)
        axw.axvline(cur, color=cur_c, linewidth=1.2)
        axw.tick_params(axis="x", colors=fg, labelsize=8, labeltop=(r == 0), labelbottom=(r == n - 1),
                        top=(r == 0), bottom=(r == n - 1))
        axw.set_xlim(0, end)
        if r == 0:
            axw.xaxis.set_ticks_position("both")
            axw.text(cur, 1.25, f"{cur:g} {unit}", color=cur_c, ha=("right" if cur > 0.9 * end else "center"), va="bottom", fontsize=8)
    items_last = fig.axes[-1]
    items_last.set_xlabel("time (" + unit + ")", color=fg, fontsize=9)
    plt.show()

## Part A: Verilog testbenches

## Step 2: A design and its testbench
A **testbench** is a module with no inputs or outputs. It:
1. creates the clock and the inputs,
2. connects your design (called the **DUT**, device under test),
3. checks the outputs and prints PASS or FAIL.

Here is a 4-bit counter. Run the cell to save the code, then read the testbench in the next cell.

In [ ]:
counter = """
module counter(input logic clk, input logic reset, output logic [3:0] count);
    always_ff @(posedge clk) begin
        if (reset) count <= 4'd0;
        else       count <= count + 4'd1;
    end
endmodule
"""

**Read this testbench before you run it.** Leave the `$dumpfile` and `$dumpvars` lines alone. They save the waveform.

Things to spot:
- `always #5 clk = ~clk;` flips the clock every 5 time units.
- `expected` is what the counter *should* show. The testbench compares it with `count` after every clock edge.
- `!==` is a strict compare that also catches `x` (unknown) values.

In [ ]:
counter_tb = """
`timescale 1ns/1ps
module tb;
    reg clk = 0;
    reg reset = 1;
    wire [3:0] count;
    integer errors = 0;
    integer i;
    reg [3:0] expected;

    counter dut(.clk(clk), .reset(reset), .count(count));

    always #5 clk = ~clk;

    initial begin
        $dumpfile("build/dump.vcd");
        $dumpvars(0, tb);
    end

    initial begin
        repeat (2) @(posedge clk);
        #1 reset = 0;
        expected = 0;
        for (i = 0; i < 20; i = i + 1) begin
            @(posedge clk); #1;
            expected = expected + 1;
            if (count !== expected) begin
                $display("FAIL at %0t: count=%0d expected=%0d", $time, count, expected);
                errors = errors + 1;
            end
        end
        if (errors == 0) $display("PASS: counter counted correctly");
        else             $display("FAILED with %0d errors", errors);
        $finish;
    end
endmodule
"""
vcd = run_sim(counter, counter_tb, name="counter")

## Step 3: See the waveform
Read it left to right. `reset` starts high, so `count` stays at 0. When `reset` drops, `count` goes up by 1 on every rising clock edge and wraps from 15 back to 0.

In [ ]:
show_waves("build/dump.vcd", ["clk", "reset", "count"])

## Step 4: Find the bug (team challenge)
This counter has one mistake. **Run it**, then use the FAIL messages and the waveform to find the bug.

Discuss as a team:
- What does the first FAIL line say?
- Where in the waveform does `count` first differ from what you expect?
- Which line of the design causes it? Fix it in the cell and run it again until you see PASS.

In [ ]:
buggy_counter = """
module counter(input logic clk, input logic reset, output logic [3:0] count);
    always_ff @(posedge clk) begin
        if (reset) count <= 4'd1;
        else       count <= count + 4'd1;
    end
endmodule
"""
vcd = run_sim(buggy_counter, counter_tb, name="buggy")
show_waves("build/dump.vcd", ["clk", "reset", "count"])

## Step 5: An ALU testbench
Not everything has a clock. An ALU is **combinational**: the output changes whenever the inputs change. The testbench sets inputs, waits a moment, and checks the output.

A `task` is a small reusable block. `check(...)` sets the inputs, waits, and compares `y` with the expected answer.

In [ ]:
alu = """
module alu(input logic [7:0] a, input logic [7:0] b, input logic [1:0] op,
           output logic [7:0] y, output logic zero);
    always_comb begin
        case (op)
            2'd0: y = a + b;
            2'd1: y = a - b;
            2'd2: y = a & b;
            2'd3: y = a | b;
            default: y = 8'd0;
        endcase
    end
    assign zero = (y == 8'd0);
endmodule
"""

alu_tb = """
`timescale 1ns/1ps
module tb;
    reg  [7:0] a = 0;
    reg  [7:0] b = 0;
    reg  [1:0] op = 0;
    wire [7:0] y;
    wire zero;
    integer errors = 0;

    alu dut(.a(a), .b(b), .op(op), .y(y), .zero(zero));

    initial begin
        $dumpfile("build/dump.vcd");
        $dumpvars(0, tb);
    end

    task check(input [7:0] ia, input [7:0] ib, input [1:0] iop, input [7:0] iexp);
        begin
            a = ia; b = ib; op = iop;
            #10;
            if (y !== iexp) begin
                $display("FAIL: a=%0d b=%0d op=%0d -> y=%0d, expected %0d", a, b, op, y, iexp);
                errors = errors + 1;
            end
        end
    endtask

    initial begin
        #10;
        check(8'd3,   8'd4,   2'd0, 8'd7);
        check(8'd10,  8'd3,   2'd1, 8'd7);
        check(8'hF0,  8'h3C,  2'd2, 8'h30);
        check(8'hF0,  8'h0F,  2'd3, 8'hFF);
        check(8'd5,   8'd5,   2'd1, 8'd0);
        if (errors == 0) $display("PASS: all ALU checks passed");
        else             $display("FAILED with %0d errors", errors);
        $finish;
    end
endmodule
"""
vcd = run_sim(alu, alu_tb, name="alu")
show_waves("build/dump.vcd", ["op", "a", "b", "y", "zero"])

## Part B: The same counter with a CocoTB (Python) testbench
CocoTB lets you write the testbench in Python. The design is the same Verilog counter.

Things to spot in the test:
- `wave_recorder.start(dut)` records the waveform. **Keep it as the first line.**
- `await RisingEdge(dut.clk)` means "wait for the next rising clock edge".
- `dut.reset.value = 1` sets an input. `dut.count.value` reads an output.
- `assert` stops the test and prints the message if the check is false.

In [ ]:
counter_sv = """
module counter(input logic clk, input logic reset, output logic [3:0] count);
    always_ff @(posedge clk) begin
        if (reset) count <= 4'd0;
        else       count <= count + 4'd1;
    end
endmodule
"""

counter_test = """
import cocotb
import wave_recorder
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge, Timer

@cocotb.test()
async def test_counter_counts(dut):
    wave_recorder.start(dut)
    cocotb.start_soon(Clock(dut.clk, 10, "ns").start())

    # hold reset for 2 clock edges, then release it
    dut.reset.value = 1
    await RisingEdge(dut.clk)
    await RisingEdge(dut.clk)
    await Timer(1, "ns")
    dut.reset.value = 0

    # after each edge the count should go up by 1 (and wrap from 15 to 0)
    expected = 0
    for i in range(20):
        await RisingEdge(dut.clk)
        await Timer(1, "ns")
        expected = (expected + 1) % 16
        actual = int(dut.count.value)
        assert actual == expected, f"cycle {i}: count={actual} expected={expected}"
"""

wave = run_cocotb(counter_sv, counter_test, top="counter", name="counter")

You should see `RESULT: PASS (1 of 1 tests passed)`. Now draw the waveform:

In [ ]:
if wave:
    show_waves(wave, ["clk", "reset", "count"])

### CocoTB bug challenge
Same test, but this design has a bug. Run it, read the failure message, and use the waveform to find the bug. Fix the design and run again until you see PASS.

In [ ]:
buggy_sv = """
module counter(input logic clk, input logic reset, output logic [3:0] count);
    always_ff @(posedge clk) begin
        if (reset) count <= 4'd1;
        else       count <= count + 4'd1;
    end
endmodule
"""

wave = run_cocotb(buggy_sv, counter_test, top="counter", name="buggy")
if wave:
    show_waves(wave, ["clk", "reset", "count"])

**Your turn:**
1. Add one more `check(...)` line for a case you think could break the ALU, for example `255 + 1` (what should the 8-bit answer be?).
2. Change `2'd2: y = a & b;` to `a ^ b` in the design and run again. Does a check fail? Which one?
3. Add an XOR operation by making `op` 3 bits wide. What else needs to change?

## Wrap-up
- A testbench drives inputs, checks outputs, and prints PASS or FAIL.
- The waveform shows *when* things happened, which the PASS/FAIL line cannot.
- The same idea carries over to CocoTB (Python testbenches) and to the Vivado simulator.